In [ ]:
!pip install wandb -q
!pip install timm -q

### Imports

In [ ]:
import contextlib
import os
import shutil
import time
import warnings
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import wandb
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             roc_auc_score)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from tqdm import tqdm

# Reproducibility and Warnings
torch.manual_seed(42)
np.random.seed(42)
warnings.filterwarnings("ignore", category=UserWarning, module='librosa')
warnings.filterwarnings("ignore", category=FutureWarning, module='librosa')

wandb.login(key="5977d6c3b044eb3d92080d4075b5683327a497ac")

### Configuration Classes

In [ ]:
@dataclass
class PathConfig:
    ljspeech_dir: Path = Path("/kaggle/input/the-lj-speech-dataset/LJSpeech-1.1")
    wavefake_dir: Path = Path("/kaggle/input/wavefake-test/generated_audio")
    analysis_output_dir: Path = Path("/kaggle/working/audio_analysis_outputs")
    manual_audio_originals_dir: Path = Path("/kaggle/working/manual_audio_originals_FOR_INSPECTION")

    def __post_init__(self):
        for p in [self.analysis_output_dir, self.manual_audio_originals_dir,
                  self.manual_audio_originals_dir / "real_aside",
                  self.manual_audio_originals_dir / "fake_aside"]:
            p.mkdir(parents=True, exist_ok=True)

@dataclass
class AudioConfig:
    SR: int = 16000; N_FFT: int = 1024; HOP_LENGTH: int = 256
    N_MELS: int = 128; FMIN: float = 0.0; FMAX: float = 8000.0
    SEGMENT_LENGTH_SECONDS: float = 3.0; NORM_EPSILON: float = 1e-6

@dataclass
class AugmentationConfig:
    apply: bool = True; num_time_masks: int = 2; num_freq_masks: int = 2
    time_mask_max_width: int = 30; freq_mask_max_width: int = 15
    mask_replacement_value: float = 0.0

@dataclass
class ModelConfig:
    name: str = "resnet18"; use_pretrained: bool = True
    img_size_cnn: int = 224; num_classes: int = 2
    normalization_strategy: str = "imagenet"
    effective_type: str = ""; effective_size: str = ""

@dataclass
class TrainingConfig:
    learning_rate: float = 5e-5; batch_size: int = 16; epochs: int = 15
    weight_decay: float = 1e-4; num_workers: int = 2
    warmup_epochs: int = 3; patience: int = 5

@dataclass
class ExperimentConfig:
    paths: PathConfig = field(default_factory=PathConfig)
    audio: AudioConfig = field(default_factory=AudioConfig)
    augmentation: AugmentationConfig = field(default_factory=AugmentationConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    SEED: int = 42; run_name: str = ""
    num_audio_to_set_aside_real: int = 100
    num_audio_to_set_aside_fake: int = 100
    run_duration_analysis: bool = False

    def validate_training_params(self):
        cfg = self.training; 
        checks = {
            'learning_rate': (lambda x: x > 0, ">0"), 'batch_size': (lambda x: x > 0, ">0"),
            'epochs': (lambda x: x > 0, ">0"), 'num_workers': (lambda x: x >= 0, ">=0")}
        for p, (chk, msg) in checks.items(): 
            assert chk(getattr(cfg,p)), f"Param '{p}' invalid: {msg}"

CURRENT_RUN_CONFIG_FOR_COLLATE: ExperimentConfig = ExperimentConfig()

### Audio Processing & Analysis

In [ ]:
# https://docs.pytorch.org/vision/stable/transforms.html#torchvision.transforms.Normalize
imagenet_normalize_transform = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def standardize_audio(path: Path | str, sr_target: int, dur_s: float) -> tuple[np.ndarray | None, int | None]:
    try:
        y, sr_orig = librosa.load(str(path), sr=None, mono=True) # Ensure path is str for librosa
        if sr_orig != sr_target: y = librosa.resample(y, orig_sr=sr_orig, target_sr=sr_target)
        len_target = int(dur_s * sr_target)
        if len(y) < len_target: y = np.pad(y, (0, len_target - len(y)), 'constant')
        else: y = y[:len_target]
        return y, sr_target
    except Exception as e: print(f"Error standardizing {path}: {e}"); return None, None

def audio_to_melspec(path: Path | str, audio_cfg: AudioConfig) -> np.ndarray | None:
    y, sr = standardize_audio(path, audio_cfg.SR, audio_cfg.SEGMENT_LENGTH_SECONDS)
    if y is None: return None
    try:
        m_spec = librosa.feature.melspectrogram(
            y=y, sr=sr, n_fft=audio_cfg.N_FFT, hop_length=audio_cfg.HOP_LENGTH,
            n_mels=audio_cfg.N_MELS,fmin=audio_cfg.FMIN,fmax=audio_cfg.FMAX)
        return librosa.power_to_db(m_spec, ref=np.max)
    # Con người có thể nghe được âm thanh trong khoảng tần số từ ~20 Hz đến ~20.000 Hz (20 kHz).
    except Exception as e: print(f"Error melspec {path}: {e}"); return None

def apply_spec_augmentation(spec: torch.Tensor, aug_cfg: AugmentationConfig) -> torch.Tensor:
    if not aug_cfg.apply: return spec
    temp_s = spec.clone()
    C, H, W = temp_s.shape
    for _ in range(aug_cfg.num_freq_masks):
        if H > aug_cfg.freq_mask_max_width:
            f = torch.randint(0, aug_cfg.freq_mask_max_width + 1, (1,)).item()
            if f > 0: f0=torch.randint(0, H - f + 1,(1,)).item(); temp_s[:,f0:f0+f,:]=aug_cfg.mask_replacement_value
    for _ in range(aug_cfg.num_time_masks):
        if W > aug_cfg.time_mask_max_width:
            t = torch.randint(0, aug_cfg.time_mask_max_width + 1, (1,)).item()
            if t > 0: t0=torch.randint(0, W - t + 1,(1,)).item(); temp_s[:,:,t0:t0+t]=aug_cfg.mask_replacement_value
    return temp_s

def preprocess_spectrogram(spec_np: np.ndarray, cfg: ExperimentConfig, training_mode: bool) -> torch.Tensor | None:
    try:
        spec_tensor = torch.from_numpy(spec_np).float()
        audio_cfg, model_cfg = cfg.audio, cfg.model
        
        if spec_tensor.ndim == 2: spec_tensor = spec_tensor.unsqueeze(0)
        
        processed_s = F.interpolate(spec_tensor.unsqueeze(0),
            size=(model_cfg.img_size_cnn, model_cfg.img_size_cnn),
            mode="bilinear", align_corners=False).squeeze(0)

        if model_cfg.normalization_strategy == "imagenet":
            s_min,s_max=processed_s.min(),processed_s.max()
            norm_01=(processed_s-s_min)/(s_max-s_min+audio_cfg.NORM_EPSILON)
            norm_3c=norm_01.repeat(3,1,1) if norm_01.shape[0]==1 else norm_01
            if norm_3c.shape[0] != 3: raise ValueError(f"Expected 3 ch for ImageNet, got {norm_3c.shape[0]}")
            processed_s = imagenet_normalize_transform(norm_3c)
        elif model_cfg.normalization_strategy == "global":
            processed_s=(processed_s-processed_s.mean())/processed_s.std().clamp(min=audio_cfg.NORM_EPSILON)
            if processed_s.shape[0]==1: processed_s=processed_s.repeat(3,1,1)
        elif model_cfg.normalization_strategy != "none" and processed_s.shape[0]==1:
             processed_s=processed_s.repeat(3,1,1)
        elif processed_s.shape[0]==1: processed_s=processed_s.repeat(3,1,1)

        if training_mode:
            processed_s = apply_spec_augmentation(processed_s, cfg.augmentation)
        return processed_s
    except Exception as e: print(f"Error in preprocess_spectrogram: {e}"); return None

def run_initial_audio_analysis(path_cfg: PathConfig, all_r: list, all_f: list):
    print("\n--- Initial Audio Analysis ---")
    if not all_r and not all_f: print("No files for analysis."); return
    r_d = [librosa.get_duration(path=p) for p in tqdm(all_r, desc="RealDur", leave=False) if Path(p).exists()]
    f_d = [librosa.get_duration(path=p) for p in tqdm(all_f, desc="FakeDur", leave=False) if Path(p).exists()]
    if r_d: print(f"Real(N={len(r_d)}):Min={np.min(r_d):.2f}s,Max={np.max(r_d):.2f}s,Mean={np.mean(r_d):.2f}s")
    if f_d: print(f"Fake(N={len(f_d)}):Min={np.min(f_d):.2f}s,Max={np.max(f_d):.2f}s,Mean={np.mean(f_d):.2f}s")
    print("--- Audio Analysis Done ---")

def collect_filepaths(path_cfg: PathConfig) -> tuple[list[Path], list[Path]]:
    real_p: list[Path] = []
    lj_meta = path_cfg.ljspeech_dir / "metadata.csv"
    if lj_meta.exists():
        try:
            df = pd.read_csv(lj_meta, sep='|', header=None, quoting=3, names=['id','t1','t2'])
            real_p = [path_cfg.ljspeech_dir/"wavs"/f"{r['id']}.wav" for _,r in df.iterrows() if (path_cfg.ljspeech_dir/"wavs"/f"{r['id']}.wav").exists()]
        except Exception as e: print(f"Err LJS meta: {e}")
    fake_p = list(path_cfg.wavefake_dir.rglob('*.wav')) if path_cfg.wavefake_dir.exists() else []
    print(f"Collected: {len(real_p)} Real, {len(fake_p)} Fake audio files.")
    return real_p, fake_p

def create_main_df(r_files: list[Path], f_files: list[Path], seed: int) -> pd.DataFrame:
    if not r_files and not f_files: return pd.DataFrame(columns=['filepath', 'label'])
    r_str, f_str = [str(p) for p in r_files], [str(p) for p in f_files]
    nr, nf = len(r_str), len(f_str)
    sel_r, sel_f = r_str, f_str
    np.random.seed(seed)
    if nf > nr and nr > 0: sel_f = np.random.choice(f_str, nr, replace=False).tolist()
    elif nr > nf and nf > 0: sel_r = np.random.choice(r_str, nf, replace=False).tolist()
    print(f"  Using {len(sel_r)} real and {len(sel_f)} fake files.")
    data = [{'filepath': p,'label':1} for p in sel_r] + [{'filepath':p,'label':0} for p in sel_f]
    df = pd.DataFrame(data)
    return df.sample(frac=1,random_state=seed).reset_index(drop=True) if not df.empty else df

def export_aside_audio(p_cfg: PathConfig, n_r: int, n_f: int, all_r: list[Path], all_f: list[Path], seed: int) -> tuple[list[Path], list[Path]]:
    r_aside_p, f_aside_p = p_cfg.manual_audio_originals_dir/"real_aside", p_cfg.manual_audio_originals_dir/"fake_aside"
    meta_p = p_cfg.manual_audio_originals_dir/"aside_audio_export_metadata.csv"
    print(f"\n--- Exporting {n_r}R & {n_f}F Originals Aside ---")
    if meta_p.exists():
        try:
            df_m=pd.read_csv(meta_p);
            r_as=[Path(p) for p in df_m[df_m['type']=='real']['original_path']]; f_as=[Path(p) for p in df_m[df_m['type']=='fake']['original_path']]
            print(f"  Loaded {len(r_as)}R, {len(f_as)}F from meta. Skip."); return r_as,f_as
        except Exception as e: print(f"  Err reading aside_meta: {e}.")
    np.random.seed(seed)
    sel_r=np.random.choice(all_r,min(len(all_r),n_r),replace=False).tolist() if all_r else []
    sel_f=np.random.choice(all_f,min(len(all_f),n_f),replace=False).tolist() if all_f else []
    export_meta_list=[]
    for i,p_obj in enumerate(tqdm(sel_r,desc="Copy R Aside",leave=False)): shutil.copy2(str(p_obj),r_aside_p/f"r_{i}_{p_obj.name}"); export_meta_list.append({'original_path':str(p_obj),'type':'real'})
    for i,p_obj in enumerate(tqdm(sel_f,desc="Copy F Aside",leave=False)): shutil.copy2(str(p_obj),f_aside_p/f"f_{i}_{p_obj.name}"); export_meta_list.append({'original_path':str(p_obj),'type':'fake'})
    if export_meta_list: pd.DataFrame(export_meta_list).to_csv(meta_p,index=False)
    print(f"  Originals aside: {len(sel_r)}R, {len(sel_f)}F."); return sel_r,sel_f

### Dataset, Models, Train/Eval

In [ ]:
class AudioDataset(Dataset):
    def __init__(self, df: pd.DataFrame, config: ExperimentConfig, set_type: str):
        self.df, self.config, self.is_train = df, config, (set_type == "train")
    def __len__(self) -> int: return len(self.df)
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor] | None:
        row = self.df.iloc[idx]
        try:
            spec_np = audio_to_melspec(str(row['filepath']), self.config.audio)
            if spec_np is None: return None
            proc_spec = preprocess_spectrogram(spec_np, self.config, self.is_train)
            if proc_spec is None: return None
            return proc_spec, torch.tensor(int(row['label'])).long()
        except Exception as e: print(f"Error in getitem {row['filepath']}: {e}"); return None

def collate_fn(batch: list) -> tuple[torch.Tensor, torch.Tensor]:
    batch = [item for item in batch if item is not None]
    if not batch:
        cfg = CURRENT_RUN_CONFIG_FOR_COLLATE
        shape = (0, 3, cfg.model.img_size_cnn, cfg.model.img_size_cnn)
        return torch.empty(shape), torch.empty(0,dtype=torch.long) # INT
    specs, labels = zip(*batch); 
    return torch.stack(specs,0),torch.stack(labels,0)

def get_model(cfg: ExperimentConfig) -> nn.Module:
    m_cfg = cfg.model
    m_name = m_cfg.name
    m_name_lc = m_name.lower() 
    in_chans = 3
    
    try:
        print(f"Attempting to load timm model: '{m_name}'")
        model = timm.create_model(
            m_name,
            pretrained=m_cfg.use_pretrained,
            num_classes=m_cfg.num_classes,
            in_chans=in_chans
        )
        
        effective_size = m_name.split('.')[0].upper().replace('_', ' ')
        m_cfg.effective_size = effective_size
        
        if 'resnet' in m_name_lc:
            m_cfg.effective_type = "cnn_timm"
        elif 'vit' in m_name_lc:
            m_cfg.effective_type = "transformer_or_hybrid_timm"
        else:
            m_cfg.effective_type = "timm_generic"

        print(f"Successfully loaded '{m_name}' as '{m_cfg.effective_size}'")
        return model

    except Exception as e:
        print(f"FATAL: Failed to load model '{m_name}' with timm: {e}. Falling back to ResNet18 as a safety measure.")
        cfg.model.name = "resnet18"
        cfg.model.use_pretrained = True
        cfg.model.normalization_strategy = "imagenet"
        cfg.model.img_size_cnn = 224
        return get_model(cfg)

def train_eval_loop(model: nn.Module, loader: DataLoader, optimizer: optim.Optimizer|None,
                    criterion: nn.Module, device: torch.device, cfg: ExperimentConfig,
                    epoch_idx: int, is_train: bool) -> tuple:
    model.train(is_train)
    loss_total, batches_count = 0.0, 0
    preds_all, labels_all, probs_all = [], [], []
    t_cfg = cfg.training

    if is_train and epoch_idx < t_cfg.warmup_epochs and optimizer:
        scale_lr = min(1., (epoch_idx+1)/max(1,t_cfg.warmup_epochs))
        for grp in optimizer.param_groups: grp['lr'] = t_cfg.learning_rate * scale_lr
    
    ctx_mgr = contextlib.nullcontext() if is_train else torch.no_grad()
    loop_desc = f"E{epoch_idx+1}/{t_cfg.epochs} [{'TRN' if is_train else 'VAL'}]"

    with ctx_mgr:
        for inputs_b, targets_b in tqdm(loader, desc=loop_desc, leave=False):
            if inputs_b.nelement()==0: continue
            inputs_b,targets_b = inputs_b.to(device), targets_b.to(device)
            if is_train and optimizer: optimizer.zero_grad()

            logits_b = model(inputs_b)
            loss = criterion(logits_b, targets_b)
            if torch.isnan(loss) or torch.isinf(loss): continue

            if is_train and optimizer:
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
            
            loss_total+=loss.item(); batches_count+=1
            probs_b=torch.softmax(logits_b,dim=1); _,preds_b=torch.max(logits_b,1)
            preds_all.extend(preds_b.cpu().numpy()); labels_all.extend(targets_b.cpu().numpy())
            if not is_train: probs_all.extend(probs_b.cpu().numpy())
    
    avg_l = loss_total/batches_count if batches_count > 0 else 0.
    acc = accuracy_score(labels_all,preds_all) if labels_all and preds_all else 0.
    if is_train: return avg_l, acc
    
    cm_res = confusion_matrix(labels_all,preds_all,labels=[0,1]) if labels_all and preds_all else np.zeros((2,2),dtype=int)
    probs_arr = np.array(probs_all) if probs_all else np.array([])
    return avg_l, labels_all, preds_all, probs_arr, cm_res, acc

def orchestrate_training(model: nn.Module, tr_loader: DataLoader, vl_loader: DataLoader|None,
                         optimizer: optim.Optimizer, criterion: nn.Module,
                         device: torch.device, cfg: ExperimentConfig) -> nn.Module:
    t_cfg = cfg.training; best_f1_val = -1.0; patience_cnt = 0
    sched_tmax = max(1,t_cfg.epochs-t_cfg.warmup_epochs) if t_cfg.epochs > t_cfg.warmup_epochs else 1
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=sched_tmax,eta_min=1e-7)
    time_train_start = time.time()

    for ep in range(t_cfg.epochs):
        time_ep_start = time.time()
        avg_l_tr, acc_tr = train_eval_loop(model,tr_loader,optimizer,criterion,device,cfg,ep,True)
        if ep >= t_cfg.warmup_epochs: scheduler.step()
        dur_ep = time.time()-time_ep_start
        
        log_wdb = {"ep":ep+1,"tr_loss":avg_l_tr,"tr_acc":acc_tr,
                   "lr":optimizer.param_groups[0]['lr'],"ep_dur_s":dur_ep}
        f1_val_curr = 0.0
        if vl_loader and hasattr(vl_loader,'dataset') and vl_loader.dataset and len(vl_loader.dataset)>0:
            avg_l_vl, lbls_vl, preds_vl, probs_vl, _, acc_vl = train_eval_loop(
                model,vl_loader,None,criterion,device,cfg,ep,False)
            if lbls_vl and preds_vl and len(lbls_vl) > 0:
                 f1_val_curr=f1_score(lbls_vl,preds_vl,average="binary",zero_division=0)
            roc_vl=0.0
            if lbls_vl and len(lbls_vl)>1 and len(np.unique(lbls_vl))>1 and \
               probs_vl.size>0 and probs_vl.shape[1]==cfg.model.num_classes:
                try: roc_vl=roc_auc_score(lbls_vl,probs_vl[:,1])
                except ValueError: pass
            print(f"E{ep+1} TrL:{avg_l_tr:.4f} TrAcc:{acc_tr:.4f} | VlL:{avg_l_vl:.4f} "
                  f"VlAcc:{acc_vl:.4f} VlF1:{f1_val_curr:.4f} VlROC:{roc_vl:.4f} ({dur_ep:.2f}s)")
            log_wdb.update({"vl_loss":avg_l_vl,"vl_f1":f1_val_curr,"vl_acc":acc_vl,"vl_roc":roc_vl})
            if f1_val_curr > best_f1_val:
                best_f1_val=f1_val_curr; patience_cnt=0
                save_p = Path(f"best_model_{cfg.run_name}.pth")
                torch.save({'model_state_dict':model.state_dict(), 
                            'model_name': cfg.model.name, 'num_classes': cfg.model.num_classes,
                            'img_size_cnn': cfg.model.img_size_cnn},save_p)
                print(f"  Best model -> {save_p} (ValF1:{best_f1_val:.4f})")
                if wandb.run: wandb.summary["best_val_f1"]=best_f1_val
            else: patience_cnt+=1
            if patience_cnt>=t_cfg.patience: print("Early stopping."); break
        else: print(f"E{ep+1} TrL:{avg_l_tr:.4f} TrAcc:{acc_tr:.4f} ({dur_ep:.2f}s, No val)")
        if wandb.run: wandb.log(log_wdb)
    
    time_total_train = time.time()-time_train_start
    print(f"Total training time: {time_total_train:.2f}s")
    if wandb.run: wandb.summary["total_train_dur_s"]=time_total_train
    return model

def save_cm_plot_final(cmtx, run_name_suffix: str, dir_save: Path, prefix_title: str="CM") -> Path | None:
    dir_save.mkdir(parents=True,exist_ok=True); fig,ax=plt.subplots(figsize=(7,5))
    sns.heatmap(cmtx,annot=True,fmt="d",cmap="GnBu",xticklabels=["Fake","Real"],yticklabels=["Fake","Real"],ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{prefix_title} - {run_name_suffix}")
    path_plot = dir_save/f"cm_{run_name_suffix}.png"
    try: fig.savefig(path_plot,dpi=120,bbox_inches="tight"); plt.close(fig); return path_plot
    except Exception as e: print(f"Err save CM: {e}"); plt.close(fig); return None

### Main Orchestration Logic

In [ ]:
def initial_data_setup(main_cfg: ExperimentConfig) -> tuple[list[Path], list[Path]]:
    r_all, f_all = collect_filepaths(main_cfg.paths)
    if main_cfg.run_duration_analysis: run_initial_audio_analysis(main_cfg.paths, r_all, f_all)
    r_aside, f_aside = export_aside_audio(main_cfg.paths, main_cfg.num_audio_to_set_aside_real,
                                          main_cfg.num_audio_to_set_aside_fake, r_all, f_all, main_cfg.SEED)
    r_avail = [p for p in r_all if p not in r_aside]
    f_avail = [p for p in f_all if p not in f_aside]
    print(f"Files for training: {len(r_avail)} Real, {len(f_avail)} Fake.")
    return r_avail, f_avail

def execute_run(cfg_run: ExperimentConfig, pool_r: list[Path], pool_f: list[Path]) -> nn.Module | None:
    global CURRENT_RUN_CONFIG_FOR_COLLATE
    CURRENT_RUN_CONFIG_FOR_COLLATE = cfg_run
    torch.manual_seed(cfg_run.SEED); np.random.seed(cfg_run.SEED)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    m = get_model(cfg_run)
    cfg_run.run_name = f"{cfg_run.model.effective_size}_{datetime.now().strftime('%y%m%d_%H%M%S')}"
    print(f"\n{'='*20} Training: {cfg_run.run_name} on {dev} {'='*20}")

    df = create_main_df(pool_r, pool_f, cfg_run.SEED)
    if df.empty: print("CRIT: Main DataFrame empty. Abort."); return None
    lbls_split = df['label']
    strat_main = lbls_split if len(lbls_split.unique())==cfg_run.model.num_classes and all(c>=2 for c in lbls_split.value_counts()) else None
    df_tr_vl, df_te = train_test_split(df,test_size=0.2,random_state=cfg_run.SEED,stratify=strat_main)
    strat_tv = None
    if not df_tr_vl.empty:
        lbls_tv = df_tr_vl['label']
        if len(lbls_tv.unique())==cfg_run.model.num_classes and all(c>=2 for c in lbls_tv.value_counts()): strat_tv=lbls_tv
    df_tr,df_vl = train_test_split(df_tr_vl,test_size=0.2,random_state=cfg_run.SEED,stratify=strat_tv) if not df_tr_vl.empty else (pd.DataFrame(),pd.DataFrame())
    print(f"  Splits: Tr={len(df_tr)}, Vl={len(df_vl if df_vl is not None else [])}, Te={len(df_te if df_te is not None else [])}")
    if df_tr.empty: print("CRIT: Train set empty. Abort."); return None

    ds_tr = AudioDataset(df_tr, cfg_run, "train")
    ds_vl = AudioDataset(df_vl, cfg_run, "val") if df_vl is not None and not df_vl.empty else None
    ds_te = AudioDataset(df_te, cfg_run, "test") if df_te is not None and not df_te.empty else None
    b_size, n_work = cfg_run.training.batch_size, cfg_run.training.num_workers
    dl_tr = DataLoader(ds_tr,b_size,shuffle=True,num_workers=n_work,pin_memory=True,collate_fn=collate_fn,drop_last=(len(ds_tr)>b_size)) if ds_tr else None
    dl_vl = DataLoader(ds_vl,b_size,shuffle=False,num_workers=n_work,pin_memory=True,collate_fn=collate_fn) if ds_vl else None
    dl_te = DataLoader(ds_te,b_size,shuffle=False,num_workers=n_work,pin_memory=True,collate_fn=collate_fn) if ds_te else None
    if not dl_tr: print("CRIT: TrainLoader empty. Abort."); return None

    m.to(dev)
    params_trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  Model '{cfg_run.model.effective_size}' ready: {params_trainable:,} trainable params.")

    w_cls = torch.ones(cfg_run.model.num_classes,dtype=torch.float,device=dev)
    if not df_tr.empty: # Use DataFrame for class weights
        train_labels_from_df = df_tr['label'].astype(int).tolist()
        if train_labels_from_df:
            cts = np.bincount(train_labels_from_df, minlength=cfg_run.model.num_classes)
            if np.all(cts > 0):
                w_cpu = torch.tensor([1./max(c, 1e-9) for c in cts], dtype=torch.float)
                w_cls = (w_cpu / (w_cpu.sum() / cfg_run.model.num_classes)).to(dev)
    print(f"  Class weights (from DataFrame): {w_cls.cpu().numpy()}")
    
    crit = nn.CrossEntropyLoss(weight=w_cls).to(dev)
    optim_adamw = torch.optim.AdamW(m.parameters(),lr=cfg_run.training.learning_rate,weight_decay=cfg_run.training.weight_decay)

    cfg_wdb = {f"path_{k}":str(v) if isinstance(v,Path) else v for k,v in cfg_run.paths.__dict__.items()}
    for grp_cfg, grp_name in [(cfg_run.audio,"audio"),(cfg_run.augmentation,"aug"),(cfg_run.model,"model"),(cfg_run.training,"train")]:
        cfg_wdb.update({f"{grp_name}_{k}":v for k,v in grp_cfg.__dict__.items()})
    cfg_wdb.update({"SEED":cfg_run.SEED,"run_name_gen":cfg_run.run_name, "ds_tr_n":len(df_tr),
                    "ds_vl_n":len(df_vl if df_vl is not None else []),"ds_te_n":len(df_te if df_te is not None else [])})
    
    if wandb.run: wandb.finish()
    wandb.init(project="audio-resnet18-maxvitnano-compare",name=cfg_run.run_name,config=cfg_wdb,resume="allow",reinit=True)
    if hasattr(m,'parameters') and any(p.requires_grad for p in m.parameters()):
        wandb.watch(m,crit,log="all",log_freq=max(100,len(dl_tr)//3 if dl_tr else 100))
    
    m_trained = orchestrate_training(m,dl_tr,dl_vl,optim_adamw,crit,dev,cfg_run)
    
    print(f"\n--- Final Test Eval for {cfg_run.run_name} ---")
    if dl_te and ds_te:
        l_te,lbl_te,prd_te,prb_te,cmtx_te,acc_te = train_eval_loop(m_trained,dl_te,None,crit,dev,cfg_run,cfg_run.training.epochs,False)
        f1_te = f1_score(lbl_te,prd_te,average="binary",zero_division=0) if lbl_te and prd_te and len(lbl_te)>0 else 0.
        roc_te=0.
        if lbl_te and len(lbl_te)>1 and len(np.unique(lbl_te))>1 and prb_te.size>0 and prb_te.shape[1]==cfg_run.model.num_classes:
            try: roc_te=roc_auc_score(lbl_te,prb_te[:,1])
            except ValueError: pass
        print(f"  Test: Loss={l_te:.4f}, Acc={acc_te:.4f}, F1={f1_te:.4f}, ROC={roc_te:.4f}")
        dir_cm = cfg_run.paths.analysis_output_dir/cfg_run.model.effective_type/cfg_run.model.effective_size/"test_cm"
        p_cm = save_cm_plot_final(cmtx_te,cfg_run.run_name+"_test",dir_cm)
        log_final_wdb={"test_loss":l_te,"test_acc":acc_te,"test_f1":f1_te,"test_roc":roc_te}
        if p_cm and p_cm.exists() and wandb.run: log_final_wdb["test_cm"]=wandb.Image(str(p_cm))
        if wandb.run: wandb.log(log_final_wdb); wandb.summary.update({"final_test_f1":f1_te,"final_test_roc":roc_te})
    else: print("  No test data. Skip final test."); wandb.log({"test_status":"No test data"}) if wandb.run else None
    if wandb.run: wandb.finish()
    return m_trained

### Define Model Configs & Run

In [ ]:
MODELS_TO_RUN_SPECS = [
    {"name": "deit_tiny_patch16_224.fb_in1k", "normalization": "imagenet", "img_size": 224},
    # {"name": "resnet18", "normalization": "imagenet", "img_size": 224},
    # {"name": "maxvit_nano_rw_256.sw_in1k", "normalization": "imagenet", "img_size": 256},
]

shared_training_config = TrainingConfig(
    learning_rate=5e-5, batch_size=16, epochs=15,
    warmup_epochs=3, patience=5, weight_decay=1e-4, num_workers=2
)

base_cfg = ExperimentConfig(
    paths=PathConfig(), audio=AudioConfig(), augmentation=AugmentationConfig(),
    training=shared_training_config, run_duration_analysis=False
)

available_real_train_files, available_fake_train_files = initial_data_setup(base_cfg)
trained_model_results = {}

for model_spec_dict in MODELS_TO_RUN_SPECS:
    run_specific_cfg = ExperimentConfig(
        paths=base_cfg.paths, audio=base_cfg.audio,
        augmentation=base_cfg.augmentation, training=base_cfg.training,
        SEED=base_cfg.SEED,
        num_audio_to_set_aside_real=base_cfg.num_audio_to_set_aside_real,
        num_audio_to_set_aside_fake=base_cfg.num_audio_to_set_aside_fake,
        run_duration_analysis=False,
            
        model=ModelConfig(
            name=model_spec_dict["name"],
            use_pretrained=True,
            normalization_strategy=model_spec_dict["normalization"],
            img_size_cnn=model_spec_dict["img_size"] # Set model-specific input size
        )
    )

    try: run_specific_cfg.validate_training_params()
    except AssertionError as e:
        print(f"Config validation error for {model_spec_dict['name']}: {e}. Skipping.")
        continue
    
    print(f"\n--- Preparing: {run_specific_cfg.model.name} (Norm: {run_specific_cfg.model.normalization_strategy}, ImgSize: {run_specific_cfg.model.img_size_cnn}) ---")
    print(f"    LR={run_specific_cfg.training.learning_rate}, BS={run_specific_cfg.training.batch_size}, Epochs={run_specific_cfg.training.epochs}")

    final_model = execute_run(
        run_specific_cfg, available_real_train_files, available_fake_train_files
    )
    if final_model:
        trained_model_results[run_specific_cfg.run_name] = final_model

print(f"\n{'='*40}\nAll training runs completed.")
if trained_model_results:
    print("Successfully trained models:")
    for run_name_key in trained_model_results: print(f"  - {run_name_key}")
else: print("No models were successfully trained.")
print(f"{'='*40}")